In [7]:
from __future__ import annotations

from pathlib import Path
import re
from typing import Dict, Optional


def read_simulation_time(log_file: str | Path) -> Dict[str, Optional[float]]:
    """Read simulation timing information from an md-flexible logOutput file.

    Returns a dictionary with:
    - metric: Which line was parsed ("Total wall-clock time", "Simulate", or "Total accumulated")
    - nanoseconds: Parsed time in ns after subtracting tuning time (float)
    - seconds: Parsed time in s after subtracting tuning time (float)
    - raw_nanoseconds: Parsed runtime in ns before subtraction (float)
    - raw_seconds: Parsed runtime in s before subtraction (float)
    - tuning_nanoseconds: Parsed tuning time in ns (float, defaults to 0.0 if missing)
    - tuning_seconds: Parsed tuning time in s (float, defaults to 0.0 if missing)
    - file: Absolute path to the parsed file
    """
    path = Path(log_file).expanduser().resolve()
    text = path.read_text(encoding="utf-8", errors="replace")

    patterns = [
        r"Total wall-clock time\s*:\s*([0-9]+(?:\.[0-9]+)?)\s*ns\s*\(([0-9]+(?:\.[0-9]+)?)s\)",
        r"Simulate\s*:\s*([0-9]+(?:\.[0-9]+)?)\s*ns\s*\(([0-9]+(?:\.[0-9]+)?)s\)",
        r"Total accumulated\s*:\s*([0-9]+(?:\.[0-9]+)?)\s*ns\s*\(([0-9]+(?:\.[0-9]+)?)s\)",
    ]
    labels = ["Total wall-clock time", "Simulate", "Total accumulated"]

    # Example line: "Tuning : 151624712766 ns (  151.625s)"
    tuning_pattern = r"\bTuning\s*:\s*([0-9]+(?:\.[0-9]+)?)\s*ns\s*\(\s*([0-9]+(?:\.[0-9]+)?)s\)"
    tuning_match = re.search(tuning_pattern, text)
    tuning_ns = float(tuning_match.group(1)) if tuning_match else 0.0
    tuning_s = float(tuning_match.group(2)) if tuning_match else 0.0

    for label, pattern in zip(labels, patterns):
        match = re.search(pattern, text)
        if match:
            raw_ns = float(match.group(1))
            raw_s = float(match.group(2))
            adjusted_ns = max(0.0, raw_ns - tuning_ns)
            adjusted_s = max(0.0, raw_s - tuning_s)
            return {
                "metric": label,
                "nanoseconds": adjusted_ns,
                "seconds": adjusted_s,
                "raw_nanoseconds": raw_ns,
                "raw_seconds": raw_s,
                "tuning_nanoseconds": tuning_ns,
                "tuning_seconds": tuning_s,
                "file": str(path),
            }

    raise ValueError(
        f"No known simulation-time pattern found in {path}. Expected one of: "
        "'Total wall-clock time', 'Simulate', or 'Total accumulated'."
    )


# Example:
# result = read_simulation_time(
#     "generated_inputs_rangeCheck/totalParticles_300000/sigmaRatio_0p15/countRatio_2p00/dataLayout_AoS/run_0/logOutput_155157_2.out"
# )
# print(result)


In [8]:
import pandas as pd
import numpy as np

try:
    from scipy import stats as scipy_stats
    _HAS_SCIPY = True
except Exception:
    scipy_stats = None
    _HAS_SCIPY = False


def collect_runtime_data_rayleightaylor(base_dir: str | Path = ".", debug: bool = False) -> pd.DataFrame:
    """Collect simulation times from logOutput files for rayleighTaylor benchmark.
    
    Folder structure: traversal_<name>/cellSize_<value>/logOutput_*.out
    """
    base = Path(base_dir).expanduser().resolve()
    rows = []
    files_found = 0

    for log_path in base.rglob("logOutput_*.out"):
        files_found += 1
        # Extract traversal from parent's parent folder name (e.g., "traversal_hgrid_block4")
        traversal_folder = log_path.parent.parent.name
        if not traversal_folder.startswith("traversal_"):
            if debug:
                print(f"  Skipped: parent.parent.name={traversal_folder} (not traversal_*)")
            continue
        traversal = traversal_folder.replace("traversal_", "")
        
        # Extract cell_size from parent folder name (e.g., "cellSize_0p50")
        cellsize_folder = log_path.parent.name
        if not cellsize_folder.startswith("cellSize_"):
            if debug:
                print(f"  Skipped: parent.name={cellsize_folder} (not cellSize_*)")
            continue
        cell_size_str = cellsize_folder.replace("cellSize_", "")
        cell_size = float(cell_size_str.replace("p", "."))

        try:
            timing = read_simulation_time(log_path)
        except Exception as e:
            if debug:
                print(f"  Skipped: {log_path.name} (read error: {e})")
            continue
            
        rows.append(
            {
                "traversal": traversal,
                "cell_size": cell_size,
                "seconds": timing["seconds"],
                "log_file": str(log_path),
            }
        )
        if debug:
            print(f"  ✓ Loaded: {traversal:25} @ cellSize={cell_size:.2f} → {timing['seconds']:.3f}s")

    if debug:
        print(f"\nTotal logOutput files found: {files_found}")
        print(f"Successfully parsed: {len(rows)}")

    return pd.DataFrame(rows)


def t_critical(confidence: float, dof: int) -> float:
    """Two-sided t critical value for a given central confidence and dof."""
    if dof <= 0:
        return 0.0

    p = (1.0 + confidence) / 2.0

    if _HAS_SCIPY:
        return float(scipy_stats.t.ppf(p, dof))

    z_lookup = {
        0.68: 0.994,
        0.86: 1.080,
        0.90: 1.645,
        0.95: 1.960,
        0.99: 2.576,
    }
    return z_lookup.get(round(confidence, 2), 1.96)


def format_time_hhhmmss(seconds: float) -> str:
    """Format seconds into hh:mm:ss.ss format."""
    total_seconds = seconds
    hours = int(total_seconds // 3600)
    remaining = total_seconds % 3600
    minutes = int(remaining // 60)
    secs = remaining % 60
    return f"{hours:02d}:{minutes:02d}:{secs:05.2f}"


In [9]:

# Collect runtime data from the notebook's directory
import os
notebook_dir = Path(__file__).parent if "__file__" in dir() else Path.cwd()

runtime_df = collect_runtime_data_rayleightaylor(notebook_dir, debug=False)

if runtime_df.empty:
    print(f"No logOutput_*.out files found in {notebook_dir}")
else:
    print("Data collected:")
    print(f"  Traversals: {sorted(runtime_df['traversal'].unique().tolist())}")
    print(f"  Cell sizes: {sorted(runtime_df['cell_size'].unique().tolist())}")
    print(f"  Total measurements: {len(runtime_df)}")
    print()
    
    # Group by traversal, cell_size and compute statistics
    summary = runtime_df.groupby(["traversal", "cell_size"], as_index=False).agg(
        seconds_mean=("seconds", "mean"),
        n=("seconds", "count")
    ).sort_values(["traversal", "cell_size"])
    
    # Add column for runtime × 10,000 in hh:mm:ss.ss format
    summary["scaled_time_10k"] = (summary["seconds_mean"] * 10000).apply(format_time_hhhmmss)
    
    # Display results
    print("Runtime summary:")
    print("-" * 90)
    display_cols = ["traversal", "cell_size", "n", "seconds_mean", "scaled_time_10k"]
    result_table = summary[display_cols].copy()
    result_table.columns = ["Traversal", "Cell Size", "N", "Mean (s)", "Time × 10,000 (hh:mm:ss.ss)"]
    print(result_table.to_string(index=False))
    print("-" * 90)


Data collected:
  Traversals: ['hgrid_block4', 'hgrid_block8', 'hgrid_matching', 'lc_c08']
  Cell sizes: [1.0]
  Total measurements: 4

Runtime summary:
------------------------------------------------------------------------------------------
     Traversal  Cell Size  N  Mean (s) Time × 10,000 (hh:mm:ss.ss)
  hgrid_block4        1.0  1 22448.980              62358:16:40.00
  hgrid_block8        1.0  1 23592.036              65533:26:00.00
hgrid_matching        1.0  1 21428.723              59524:13:50.00
        lc_c08        1.0  1 22711.049              63086:14:50.00
------------------------------------------------------------------------------------------


###  2/3 in each dimension and full 600 000 iterations ###


In [ ]:
# Display condensed summary table as DataFrame
if runtime_df.empty:
    print("runtime_df is empty. Run the previous cell first.")
else:
    # Build one row per traversal with requested time columns.
    summary_for_table = summary[["traversal", "cell_size", "seconds_mean"]].copy()

    pivot = summary_for_table.pivot_table(
        index="traversal",
        columns="cell_size",
        values="seconds_mean",
        aggfunc="mean"
    )

    # Keep only cell size 1 and compute speedup relative to lc_c08.
    final_summary = pivot.copy()
    final_summary = final_summary.reindex(columns=[1.0])
    final_summary = final_summary.rename(columns={1.0: "Cell Size 1"})

    baseline_seconds = None
    if "lc_c08" in final_summary.index and "Cell Size 1" in final_summary.columns:
        baseline_seconds = final_summary.loc["lc_c08", "Cell Size 1"]

    if baseline_seconds is not None and not pd.isna(baseline_seconds):
        final_summary["Speedup vs lc_c08"] = baseline_seconds / final_summary["Cell Size 1"]
    else:
        final_summary["Speedup vs lc_c08"] = np.nan

    final_summary = final_summary.sort_values("Cell Size 1", ascending=True)

    def format_hhmmss(seconds: float) -> str:
        if pd.isna(seconds):
            return "-"
        total_seconds = int(round(float(seconds)))
        hours = total_seconds // 3600
        minutes = (total_seconds % 3600) // 60
        secs = total_seconds % 60
        return f"{hours:02d}:{minutes:02d}:{secs:02d}"

    def format_speedup(value: float) -> str:
        if pd.isna(value):
            return "-"
        return f"{value:.3f}x"

    if "Cell Size 1" in final_summary.columns:
        final_summary["Cell Size 1"] = final_summary["Cell Size 1"].apply(format_hhmmss)
    if "Speedup vs lc_c08" in final_summary.columns:
        final_summary["Speedup vs lc_c08"] = final_summary["Speedup vs lc_c08"].apply(format_speedup)

    final_summary = final_summary.reset_index().rename(columns={"traversal": "Traversal"})

    print("\nFull Results Table:")
    print(final_summary.to_string(index=False))
    


Full Results Table:
     Traversal  Fastest Cell Size 1
hgrid_matching 05:57:09    05:57:09
  hgrid_block4 06:16:41    06:16:41
        lc_c08 06:18:31    06:18:31
  hgrid_block8 06:35:39    06:35:39


Full Results Table:
     Traversal Cell Size 1 Speedup vs lc_c08
hgrid_matching    05:57:09            1.060x
  hgrid_block4    06:14:09            1.012x
        lc_c08    06:18:31            1.000x
  hgrid_block8    06:33:12            0.963x
